# 예제 04: 끝단 자세 지령 — Pose Goal (self-contained)

Cartesian 공간에서 끝단(`gripper_base`)의 위치(`x, y, z`)와 방향(`roll, pitch, yaw`)을
지정해 이동하는 예제. `utils.py`를 사용하지 않고 노트북 한 파일 안에서 모든 것을 정의한다.

**학습 내용**
- `PositionConstraint`, `OrientationConstraint` 메시지 구성
- IK(역기구학) 솔버의 역할 — pose는 입력, 조인트 값은 출력
- Euler ↔ Quaternion 변환
- RViz2 `MarkerArray` 로 목표 자세를 **미리** 시각화

**이 노트북의 워크플로**

각 목표마다 두 단계로 나뉜다:

1. **미리보기 단계** — `preview_pose_target(...)` 호출. RViz2에 마커만 찍히고 로봇은 움직이지 않는다.
   사용자는 RViz를 보고 "내가 보내려는 위치/방향이 맞나?"를 확인한다.
2. **실행 단계** — `go_to_pose_goal(...)` 호출. MoveIt이 IK + 모션 플래닝을 수행하고 실제로 로봇을 이동시킨다.

원본 스크립트: `ex04_pose_goal.py`

## 실행 절차

이 노트북은 별도로 띄운 MoveIt + RViz의 `move_group` 액션 서버에 클라이언트로 붙는 방식이다. 터미널을 둘 띄워야 한다.

> ⚠ 다른 로봇용 MoveIt launch가 떠 있으면 같은 토픽으로 충돌해 RViz가 죽거나 controller_manager가 segfault할 수 있다. 시작 전에 `pgrep -af 'ros2 launch'` 로 잔존 프로세스가 없는지 확인하자.

### 터미널 1 — MoveIt + RViz 기동

```bash
source /opt/ros/jazzy/setup.bash
source ~/robot_arm/install/setup.bash
ros2 launch robot_arm_moveit_config demo.launch.xml
```

RViz가 뜨면 **`MarkerArray` Display를 추가하고 Topic을 `/pose_goal_markers` 로 설정**한다.
이 토픽으로 미리보기 마커가 발행된다.

### 터미널 2 — Jupyter 기동

```bash
source ~/venv/ros_jazzy/bin/activate
source /opt/ros/jazzy/setup.bash
source ~/robot_arm/install/setup.bash
cd ~/robot_arm/src/robotarm_tutorials/robot_arm_tutorials/robot_arm_tutorials
jupyter lab ex04_pose_goal.ipynb
```

셀을 위에서 아래로 순서대로 실행한다 (`Shift+Enter`).

### 다시 실행하고 싶을 때

- `rclpy`는 한 프로세스에서 한 번만 init할 수 있어, **2-2. `rclpy` 초기화** 셀은 `try/except`로 감싸 두 번째 실행해도 무시한다.
- 마지막 "정리" 셀까지 실행하지 않고 노트북을 닫아도 무방하다.

## 1. 로봇 상수 정의

이 값들은 `robot_arm_moveit_config/config/robot_arm.srdf` 와 일치한다.

In [1]:
PLANNING_GROUP    = 'manipulator'
REFERENCE_FRAME   = 'base_link'
END_EFFECTOR_LINK = 'gripper_base'
ARM_JOINTS        = ['joint1', 'joint2', 'joint3', 'joint4', 'joint5', 'joint6']
MARKER_TOPIC      = '/pose_goal_markers'

## 2. ROS 2 초기화와 노드 생성

`MoveGroup` 액션 클라이언트, `MarkerArray` 퍼블리셔, `joint_states` 구독자를 같은 노드에 붙인다.

### 2-1. import

In [2]:
import math
import time
import xml.etree.ElementTree as ET

import rclpy
from rclpy.node import Node
from rclpy.action import ActionClient
from rclpy.parameter_client import AsyncParameterClient

import tf_transformations

from sensor_msgs.msg import JointState
from geometry_msgs.msg import Pose, Point, Quaternion, Vector3
from std_msgs.msg import ColorRGBA
from visualization_msgs.msg import Marker, MarkerArray
from shape_msgs.msg import SolidPrimitive

from moveit_msgs.action import MoveGroup
from moveit_msgs.msg import (
    MotionPlanRequest, PlanningOptions, MoveItErrorCodes,
    Constraints, JointConstraint, PositionConstraint, OrientationConstraint,
    BoundingVolume,
)

### 2-2. `rclpy` 초기화

한 프로세스에서 한 번만 init 가능하므로, 노트북에서 이 셀을 두 번 실행해도 무시되도록 `try/except`로 감싼다.

In [3]:
try:
    rclpy.init()
except RuntimeError:
    pass  # 이미 초기화된 경우 무시

### 2-3. 노드, 액션 클라이언트, 마커 퍼블리셔, `joint_states` 구독자

In [4]:
node = Node('ex04_pose_goal_demo')
move_client = ActionClient(node, MoveGroup, 'move_action')
marker_pub = node.create_publisher(MarkerArray, MARKER_TOPIC, 10)

joint_state = {'msg': None}
node.create_subscription(
    JointState, 'joint_states',
    lambda msg: joint_state.update(msg=msg), 10,
)
node.get_logger().info('=== 예제 04 노트북 노드 생성 완료 ===')

[INFO] [1778030253.712120284] [ex04_pose_goal_demo]: === 예제 04 노트북 노드 생성 완료 ===


True

## 3. 액션 서버와 `/joint_states` 준비 대기

In [5]:
def wait_for_ready(timeout_sec: float = 30.0) -> None:
    if not move_client.wait_for_server(timeout_sec=timeout_sec):
        raise RuntimeError('MoveGroup 액션 서버 연결 실패')
    start = time.time()
    while joint_state['msg'] is None:
        rclpy.spin_once(node, timeout_sec=0.1)
        if time.time() - start > timeout_sec:
            raise RuntimeError('joint_states 수신 실패')
    node.get_logger().info('action server + /joint_states 준비됨')

wait_for_ready()

[INFO] [1778030259.059328296] [ex04_pose_goal_demo]: action server + /joint_states 준비됨


## 4. SRDF에서 `ready` / `home` 포즈 읽어오기

`move_group` 노드가 보유한 `robot_description_semantic` 파라미터에서 SRDF XML을 받아
`<group_state>` 안의 조인트 값들을 딕셔너리로 추출한다.

| 함수 | 역할 |
|---|---|
| `fetch_srdf_xml` | `move_group` 에서 SRDF 문자열 받아옴 |
| `parse_named_pose` | SRDF XML에서 group/name 의 group_state 를 dict로 파싱 |
| `load_named_pose` | 위 둘을 묶는 진입점 |

In [6]:
def fetch_srdf_xml(timeout_sec: float = 10.0) -> str:
    client = AsyncParameterClient(node, 'move_group')
    if not client.wait_for_services(timeout_sec=timeout_sec):
        raise RuntimeError('move_group 파라미터 서비스 연결 실패')
    future = client.get_parameters(['robot_description_semantic'])
    rclpy.spin_until_future_complete(node, future, timeout_sec=timeout_sec)
    return future.result().values[0].string_value

def parse_named_pose(srdf_xml: str, name: str, group: str) -> dict:
    root = ET.fromstring(srdf_xml)
    for gs in root.findall('group_state'):
        if gs.attrib.get('group') == group and gs.attrib.get('name') == name:
            return {j.attrib['name']: float(j.attrib.get('value', '0'))
                    for j in gs.findall('joint')}
    raise RuntimeError(f'SRDF group_state "{name}" (group={group}) 없음')

def load_named_pose(name: str, timeout_sec: float = 10.0) -> dict:
    return parse_named_pose(fetch_srdf_xml(timeout_sec), name, PLANNING_GROUP)

ready_target = load_named_pose('ready')
home_target  = load_named_pose('home')
node.get_logger().info(f'ready: {ready_target}')
node.get_logger().info(f'home : {home_target}')

[INFO] [1778030260.942930548] [ex04_pose_goal_demo]: ready: {'joint1': 0.0, 'joint2': 0.051, 'joint3': 0.594, 'joint4': 0.73, 'joint5': -0.051, 'joint6': 0.323}
[INFO] [1778030260.943878255] [ex04_pose_goal_demo]: home : {'joint1': 0.0, 'joint2': 0.0, 'joint3': 0.0, 'joint4': 0.0, 'joint5': 0.0, 'joint6': 0.0}


True

## 5. Pose 만들기 — Euler → Quaternion 변환

`MoveIt` 의 `Pose` 메시지는 방향을 쿼터니언(`x, y, z, w`)으로 표현한다.
사람이 입력하기 편한 `roll/pitch/yaw` (rad)을 쿼터니언으로 바꾸는 헬퍼 두 개를 만든다.

In [7]:
def euler_to_quaternion(roll: float, pitch: float, yaw: float) -> Quaternion:
    q = tf_transformations.quaternion_from_euler(roll, pitch, yaw)
    return Quaternion(x=q[0], y=q[1], z=q[2], w=q[3])

def make_pose(x: float, y: float, z: float,
              roll: float = 0.0, pitch: float = 0.0, yaw: float = 0.0) -> Pose:
    pose = Pose()
    pose.position = Point(x=x, y=y, z=z)
    pose.orientation = euler_to_quaternion(roll, pitch, yaw)
    return pose

## 6. **목표 자세 미리보기** — RViz2 마커 발행

이 노트북의 핵심. `preview_pose_target(target)` 한 번 호출 = `/pose_goal_markers` 토픽으로 **마커만** 발행.
로봇은 움직이지 않는다. RViz를 보고 "여기로 보내려는 게 맞다" 라고 사용자가 확인한 뒤에 다음 셀에서 실행한다.

발행되는 마커 3개:

| 마커 | 의미 |
|---|---|
| `ARROW`  | 끝단의 위치 + 방향 (`roll/pitch/yaw` 적용된 화살표) |
| `SPHERE` | 끝단의 위치만 강조 |
| `TEXT_VIEW_FACING` | 목표의 라벨 표시 |

색상 의미: 노란색 = 미리보기, 파란색 = 진행 중, 초록색 = 성공, 빨간색 = 실패.

In [8]:
COLOR_PENDING = ColorRGBA(r=1.0, g=1.0, b=0.0, a=0.8)   # 노란색
COLOR_ACTIVE  = ColorRGBA(r=0.2, g=0.5, b=1.0, a=0.9)   # 파란색
COLOR_SUCCESS = ColorRGBA(r=0.0, g=1.0, b=0.0, a=0.8)   # 초록색
COLOR_FAIL    = ColorRGBA(r=1.0, g=0.0, b=0.0, a=0.8)   # 빨간색
COLOR_TEXT    = ColorRGBA(r=1.0, g=1.0, b=1.0, a=1.0)   # 흰색

# 발행한 마커들의 누적 상태 (덮어쓰기 위해 보관)
_markers = MarkerArray()

def _build_markers(idx: int, target: dict, color: ColorRGBA) -> list:
    stamp = node.get_clock().now().to_msg()
    pose = make_pose(target['x'], target['y'], target['z'],
                     target['roll'], target['pitch'], target['yaw'])

    arrow = Marker()
    arrow.header.frame_id = REFERENCE_FRAME
    arrow.header.stamp = stamp
    arrow.ns = 'pose_goal_arrow'
    arrow.id = idx
    arrow.type = Marker.ARROW
    arrow.action = Marker.ADD
    arrow.pose = pose
    arrow.scale = Vector3(x=0.12, y=0.02, z=0.02)
    arrow.color = color

    sphere = Marker()
    sphere.header.frame_id = REFERENCE_FRAME
    sphere.header.stamp = stamp
    sphere.ns = 'pose_goal_sphere'
    sphere.id = idx
    sphere.type = Marker.SPHERE
    sphere.action = Marker.ADD
    sphere.pose.position = Point(x=target['x'], y=target['y'], z=target['z'])
    sphere.pose.orientation.w = 1.0
    sphere.scale = Vector3(x=0.045, y=0.045, z=0.045)
    sphere.color = ColorRGBA(r=color.r, g=color.g, b=color.b, a=0.6)

    text = Marker()
    text.header.frame_id = REFERENCE_FRAME
    text.header.stamp = stamp
    text.ns = 'pose_goal_text'
    text.id = idx
    text.type = Marker.TEXT_VIEW_FACING
    text.action = Marker.ADD
    text.pose.position = Point(x=target['x'], y=target['y'], z=target['z'] + 0.12)
    text.pose.orientation.w = 1.0
    text.scale.z = 0.05
    text.color = COLOR_TEXT
    text.text = target.get('label', f'target {idx}')
    return [arrow, sphere, text]

def publish_marker(idx: int, target: dict, color: ColorRGBA) -> None:
    """같은 (ns, id) 마커는 덮어쓰고, 누적 MarkerArray 를 한 번에 발행."""
    new = _build_markers(idx, target, color)
    keys = {(m.ns, m.id) for m in new}
    _markers.markers = [m for m in _markers.markers if (m.ns, m.id) not in keys]
    _markers.markers.extend(new)
    marker_pub.publish(_markers)

def preview_pose_target(target: dict, idx: int = 1) -> None:
    """목표 자세를 RViz2 에 노란색 미리보기 마커로 표시. 로봇은 움직이지 않는다.

    target dict: {'label': str, 'x': float, 'y': float, 'z': float,
                  'roll': rad, 'pitch': rad, 'yaw': rad}
    """
    publish_marker(idx, target, COLOR_PENDING)
    node.get_logger().info(
        f"[preview] idx={idx} {target.get('label', '')} "
        f"pos=({target['x']:.2f}, {target['y']:.2f}, {target['z']:.2f}) "
        f"rpy=({math.degrees(target['roll']):.0f}°, "
        f"{math.degrees(target['pitch']):.0f}°, "
        f"{math.degrees(target['yaw']):.0f}°)"
    )

## 7. 조인트 목표 보내기 — `ready` / `home` 이동용

`ready` 와 `home` 은 SRDF에 정의된 조인트 자세이므로 Pose 가 아닌 조인트 제약으로 보낸다.
(예제 03과 동일한 방식)

In [9]:
def make_joint_constraints(joint_values: dict, tol: float = 0.01) -> Constraints:
    constraints = Constraints()
    for jname, val in joint_values.items():
        constraints.joint_constraints.append(JointConstraint(
            joint_name=jname, position=val,
            tolerance_above=tol, tolerance_below=tol, weight=1.0,
        ))
    return constraints

def make_plan_request(vel: float, acc: float,
                       attempts: int, plan_time: float) -> MotionPlanRequest:
    req = MotionPlanRequest()
    req.group_name = PLANNING_GROUP
    req.num_planning_attempts = attempts
    req.allowed_planning_time = plan_time
    req.max_velocity_scaling_factor = vel
    req.max_acceleration_scaling_factor = acc
    return req

def make_goal(req: MotionPlanRequest) -> MoveGroup.Goal:
    goal = MoveGroup.Goal()
    goal.request = req
    goal.planning_options = PlanningOptions(
        plan_only=False, replan=True, replan_attempts=3)
    return goal

def send_goal_and_wait(goal: MoveGroup.Goal) -> int:
    send_future = move_client.send_goal_async(goal)
    rclpy.spin_until_future_complete(node, send_future)
    handle = send_future.result()
    if handle is None or not handle.accepted:
        return MoveItErrorCodes.PLANNING_FAILED
    result_future = handle.get_result_async()
    rclpy.spin_until_future_complete(node, result_future)
    return result_future.result().result.error_code.val

def go_to_joint_goal(joint_values: dict, vel: float = 0.3, acc: float = 0.3,
                      attempts: int = 5, plan_time: float = 5.0) -> bool:
    req = make_plan_request(vel, acc, attempts, plan_time)
    req.goal_constraints.append(make_joint_constraints(joint_values))
    code = send_goal_and_wait(make_goal(req))
    ok = (code == MoveItErrorCodes.SUCCESS)
    if not ok:
        node.get_logger().error(f'joint goal 실패 error_code={code}')
    return ok

## 8. 끝단 자세 목표 보내기 — Pose Goal

`Pose` 한 개를 받아 `PositionConstraint` 와 `OrientationConstraint` 두 개로 변환한다.
MoveIt 이 IK를 풀어 도달 가능한 조인트 값을 찾고 그 경로로 이동시킨다.

| 함수 | 역할 |
|---|---|
| `make_position_constraint` | 위치 허용 영역 = 작은 구(반지름 1cm) |
| `make_orientation_constraint` | 방향 허용 오차 = 1cm/축 정도 |
| `go_to_pose_goal` | 위 둘을 묶어 `Constraints` 로 만들고 액션 송신 |

In [10]:
def make_position_constraint(pose: Pose, tol: float = 0.01) -> PositionConstraint:
    pc = PositionConstraint()
    pc.header.frame_id = REFERENCE_FRAME
    pc.link_name = END_EFFECTOR_LINK
    pc.target_point_offset = Vector3(x=0.0, y=0.0, z=0.0)

    bv = BoundingVolume()
    sphere = SolidPrimitive()
    sphere.type = SolidPrimitive.SPHERE
    sphere.dimensions = [tol]
    bv.primitives.append(sphere)

    sp = Pose()
    sp.position = Point(x=pose.position.x, y=pose.position.y, z=pose.position.z)
    sp.orientation.w = 1.0
    bv.primitive_poses.append(sp)

    pc.constraint_region = bv
    pc.weight = 1.0
    return pc

def make_orientation_constraint(pose: Pose, tol: float = 0.01) -> OrientationConstraint:
    oc = OrientationConstraint()
    oc.header.frame_id = REFERENCE_FRAME
    oc.link_name = END_EFFECTOR_LINK
    oc.orientation = pose.orientation
    oc.absolute_x_axis_tolerance = tol
    oc.absolute_y_axis_tolerance = tol
    oc.absolute_z_axis_tolerance = tol
    oc.weight = 1.0
    return oc

def go_to_pose_goal(pose: Pose, vel: float = 0.3, acc: float = 0.3,
                     attempts: int = 5, plan_time: float = 10.0) -> bool:
    req = make_plan_request(vel, acc, attempts, plan_time)
    constraints = Constraints()
    constraints.position_constraints.append(make_position_constraint(pose))
    constraints.orientation_constraints.append(make_orientation_constraint(pose))
    req.goal_constraints.append(constraints)
    code = send_goal_and_wait(make_goal(req))
    ok = (code == MoveItErrorCodes.SUCCESS)
    if not ok:
        node.get_logger().error(f'pose goal 실패 error_code={code} (IK 해 없음 가능)')
    return ok

## 8-1. 도달 가능성 미리 확인 — Plan Only

`go_to_pose_goal` 은 IK + 모션 플래닝 + 실행을 한 번에 한다.
하지만 어떤 자세는 IK 해가 없거나(작업 공간 밖, 특이점), 무충돌 경로를 못 찾을 수도 있다.
실제로 보내기 전에 **계획만** 시켜보고 도달 가능 여부를 알 수 있으면 안전하다.

`PlanningOptions.plan_only=True` 로 같은 액션을 송신하면, MoveIt이 IK + 플래닝까지 수행해 trajectory 까지 계산하지만 컨트롤러로는 보내지 않는다.

| 함수 | 역할 |
|---|---|
| `check_pose_reachable(pose)` | Pose 에 IK + 플래닝만 수행 — 로봇 안 움직이고 `(ok, error_code)` 반환 |

자주 보는 `MoveItErrorCodes` 값:

| 코드 | 이름 | 의미 |
|---|---|---|
| `1` | `SUCCESS` | 경로 찾음 — 도달 가능 |
| `-1` | `PLANNING_FAILED` | 플래너가 경로를 못 찾음 |
| `-12` | `GOAL_IN_COLLISION` | 목표 자세에서 충돌 발생 |
| `-31` | `NO_IK_SOLUTION` | IK 해 없음 (작업 공간 밖) |

In [ ]:
# 자주 보는 MoveItErrorCodes 값을 사람이 읽기 좋은 이름으로 매핑
MOVEIT_ERROR_NAMES = {
      1: 'SUCCESS',
     -1: 'PLANNING_FAILED',
     -2: 'INVALID_MOTION_PLAN',
     -6: 'TIMED_OUT',
    -10: 'START_STATE_IN_COLLISION',
    -11: 'START_STATE_VIOLATES_PATH_CONSTRAINTS',
    -12: 'GOAL_IN_COLLISION',
    -13: 'GOAL_VIOLATES_PATH_CONSTRAINTS',
    -14: 'GOAL_CONSTRAINTS_VIOLATED',
    -15: 'INVALID_GROUP_NAME',
    -16: 'INVALID_GOAL_CONSTRAINTS',
    -17: 'INVALID_ROBOT_STATE',
    -27: 'GOAL_STATE_INVALID',
    -31: 'NO_IK_SOLUTION',
}

def check_pose_reachable(pose: Pose, vel: float = 0.3, acc: float = 0.3,
                          attempts: int = 5, plan_time: float = 5.0):
    """Pose 목표에 IK + 모션 플래닝만 수행한다 (로봇은 움직이지 않음).

    Returns:
        (ok: bool, error_code: int)
    """
    req = make_plan_request(vel, acc, attempts, plan_time)
    constraints = Constraints()
    constraints.position_constraints.append(make_position_constraint(pose))
    constraints.orientation_constraints.append(make_orientation_constraint(pose))
    req.goal_constraints.append(constraints)

    goal = MoveGroup.Goal()
    goal.request = req
    # plan_only=True → 계획(IK + 경로)까지만 하고 컨트롤러로는 보내지 않음
    goal.planning_options = PlanningOptions(
        plan_only=True, replan=False, replan_attempts=0)

    code = send_goal_and_wait(goal)
    ok = (code == MoveItErrorCodes.SUCCESS)
    name = MOVEIT_ERROR_NAMES.get(code, f'code={code}')
    if ok:
        node.get_logger().info('  ✓ 도달 가능 — 경로 존재 확인')
    else:
        node.get_logger().warn(f'  ✗ 도달 불가 — {name}')
    return ok, code

## 9. 시나리오 시작 — `ready` 자세로 초기화

Pose 목표를 보내기 전에 잘 보이는 자세(`ready`)로 옮긴다.

In [11]:
node.get_logger().info('--- ready 자세로 초기 이동 ---')
go_to_joint_goal(ready_target)
time.sleep(1.0)

[INFO] [1778030299.486354089] [ex04_pose_goal_demo]: --- ready 자세로 초기 이동 ---


## 10. 첫 번째 목표 — 전방 수평 자세

끝단을 로봇 전방 (`x=0.30, y=0.0, z=0.30`) 으로 보내고, 끝단이 아래를 향하도록 `roll = π` 를 준다.

### 10-1. 목표 정의 + 미리보기 (마커만 발행, 로봇은 움직이지 않음)

이 셀을 실행한 뒤 RViz에서 **노란색 화살표/구**가 의도한 위치에 있는지 확인하고 다음 셀로 넘어간다.

In [12]:
target1 = {
    'label': 'Forward',
    'x': 0.30, 'y': 0.00, 'z': 0.30,
    'roll': math.pi, 'pitch': 0.0, 'yaw': 0.0,
}
preview_pose_target(target1, idx=1)

[INFO] [1778030305.880212224] [ex04_pose_goal_demo]: [preview] idx=1 Forward pos=(0.30, 0.00, 0.30) rpy=(180°, 0°, 0°)


### 10-2. 마커 위치 확인했으면 실제로 이동

(원하면 위 셀에서 `target1` 의 값을 바꾼 뒤 미리보기를 다시 띄워보고 이 셀을 실행해도 된다.)

In [16]:
publish_marker(1, target1, COLOR_ACTIVE)
ok = go_to_pose_goal(make_pose(
    target1['x'], target1['y'], target1['z'],
    target1['roll'], target1['pitch'], target1['yaw'],
))
publish_marker(1, target1, COLOR_SUCCESS if ok else COLOR_FAIL)
time.sleep(1.0)

## 11. 두 번째 목표 — 좌측

끝단을 로봇 좌측 (`y=+0.30`) 으로 보내고 yaw 를 90° 돌려 손목을 좌측을 향하게 한다.

### 11-1. 미리보기

In [14]:
target2 = {
    'label': 'Left',
    'x': 0.00, 'y': 0.30, 'z': 0.30,
    'roll': math.pi, 'pitch': 0.0, 'yaw': math.pi / 2,
}
preview_pose_target(target2, idx=2)

[INFO] [1778030362.950323823] [ex04_pose_goal_demo]: [preview] idx=2 Left pos=(0.00, 0.30, 0.30) rpy=(180°, 0°, 90°)


### 11-2. 실제로 이동

In [34]:
publish_marker(2, target2, COLOR_ACTIVE)
ok = go_to_pose_goal(make_pose(
    target2['x'], target2['y'], target2['z'],
    target2['roll'], target2['pitch'], target2['yaw'],
))
publish_marker(2, target2, COLOR_SUCCESS if ok else COLOR_FAIL)
time.sleep(1.0)

## 12. 세 번째 목표 — 높이 올린 자세

끝단을 위로 올린 자세 (`z=0.40`).

### 12-1. 미리보기

In [18]:
target3 = {
    'label': 'High',
    'x': 0.20, 'y': 0.00, 'z': 0.40,
    'roll': math.pi, 'pitch': 0.0, 'yaw': 0.0,
}
preview_pose_target(target3, idx=3)

[INFO] [1778030411.887593163] [ex04_pose_goal_demo]: [preview] idx=3 High pos=(0.20, 0.00, 0.40) rpy=(180°, 0°, 0°)


### 12-2. 실제로 이동

In [19]:
publish_marker(3, target3, COLOR_ACTIVE)
ok = go_to_pose_goal(make_pose(
    target3['x'], target3['y'], target3['z'],
    target3['roll'], target3['pitch'], target3['yaw'],
))
publish_marker(3, target3, COLOR_SUCCESS if ok else COLOR_FAIL)
time.sleep(1.0)

## 13. 보너스 — 직접 목표를 입력해 보기

세 단계로 나뉜다:

1. `preview_pose_target(my_target)` — 마커만 표시 (로봇은 가만히)
2. `check_pose_reachable(my_pose)` — 도달 가능한지 미리 확인 (실행 X)
3. 가능하면 `go_to_pose_goal(my_pose)` — 실제 이동

`my_target` 값을 수정해 가며 작업 공간의 한계를 시험해 보자.
작업 공간 밖이면 step 2 에서 `NO_IK_SOLUTION (-31)` 등이 떠 step 3 으로 넘어가지 않게 된다.

### 13-1. 미리보기

In [ ]:
my_target = {
    'label': 'MyTarget',
    'x': 0.25, 'y': -0.15, 'z': 0.30,
    'roll': math.pi, 'pitch': 0.0, 'yaw': -math.pi / 4,
}
preview_pose_target(my_target, idx=99)

### 13-2. 도달 가능성 검사 (실행 X)

위의 `my_target` 이 정말 도달 가능한지 미리 확인한다.
`reachable=True` 면 다음 셀로 가서 이동, `False` 면 위로 올라가 `my_target` 값을 수정하고 다시 시도한다.

In [ ]:
my_pose = make_pose(
    my_target['x'], my_target['y'], my_target['z'],
    my_target['roll'], my_target['pitch'], my_target['yaw'],
)
reachable, code = check_pose_reachable(my_pose)
print(f'reachable = {reachable}, error_code = {code}')

### 13-3. 도달 가능하면 실제로 이동

In [ ]:
publish_marker(99, my_target, COLOR_ACTIVE)
ok = go_to_pose_goal(my_pose)
publish_marker(99, my_target, COLOR_SUCCESS if ok else COLOR_FAIL)

## 14. `home` 으로 복귀

In [35]:
node.get_logger().info('--- home 으로 복귀 ---')
go_to_joint_goal(home_target)
node.get_logger().info('=== 예제 04 완료! ===')

[INFO] [1778030564.702093853] [ex04_pose_goal_demo]: --- home 으로 복귀 ---
[INFO] [1778030572.260709627] [ex04_pose_goal_demo]: === 예제 04 완료! ===


True

## 15. 정리

노트북을 닫기 전에 노드와 rclpy를 안전하게 정리한다.

In [ ]:
node.destroy_node()
try:
    rclpy.shutdown()
except Exception:
    pass